# Design a lean, blinded shoreline-extraction pilot

**Status:** local sector design approved on 2026-08-07; Design 1 retrieval dates approved on 2026-08-08 in `03-choose-pilot-dates.ipynb`; the extraction method is not frozen.

This notebook turns the live Landsat availability results and the completed AIMS
review into a small, auditable pilot design. Its purpose is to choose and freeze
extraction, validation, filtering, correction and completeness rules before any
full-coast shoreline rates are inspected.

It does **not** redefine the Holderness study extent, classify whole availability
ROIs, construct final defence covariates, download imagery or extract shorelines.

## 1. Decisions the pilot must support

The canonical plan requires a blinded pilot before full-coast rates (`P009`–`P010`).
It must cover representative missions, decades, seasons, water levels, defended and
undefended settings, exposed till, ords and other extraction challenges (`P020`).

The reusable outcome is a frozen coast-wide method: classifier choice, water-level
filter, buffer and intersection settings, QC exclusions, validation evidence and
completeness rules. Pilot choices must be based on extraction validity, independent
positional accuracy and availability—not plausible-looking erosion rates (`P025`).

Availability ROIs remain operational query/retrieval units. Spatial context belongs
to short local sectors; water level belongs to individual scene dates. Final defence
history and distance-to-defence-end covariates are later transect-level work
(`P081`–`P083`).

In [ ]:
from pathlib import Path
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyogrio
from matplotlib.lines import Line2D
from shapely import get_coordinates
from shapely.geometry import Point
from shapely.ops import substring

ROOT = Path.cwd()
if not (ROOT / "implementation-plan.json").exists():
    ROOT = ROOT.parent

assert (ROOT / "implementation-plan.json").exists(), (
    f"Could not locate the repository root from {Path.cwd()}"
)

FIGURE_DIR = ROOT / "outputs/figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "rois": ROOT / "data/derived/geometry/availability-rois.geojson",
    "seed": ROOT / "data/derived/geometry/os-geometry-seed.geojson",
    "availability": ROOT / "data/derived/availability/landsat-scene-availability.csv",
    "scene_pool": ROOT / "data/derived/pilot/pilot-candidate-pool.csv",
    "scene_summary": ROOT / "data/derived/pilot/pilot-candidate-summary.json",
    "aims_inspection": ROOT / "data/interim/defences/aims-holderness-inspection.gpkg",
    "asset_review": ROOT / "data/interim/defences/aims-coastal-asset-review.csv",
    "functional_review": ROOT / "data/interim/defences/aims-holderness-functional-review.csv",
    "frontage_review": ROOT / "data/interim/defences/holderness-frontage-group-review.csv",
    "council_review": ROOT / "data/interim/defences/holderness-council-defended-location-review.csv",
    "sector_candidates": ROOT / "data/interim/pilot/pilot-sector-candidates.csv",
    "design_summary": ROOT / "data/derived/pilot/pilot-design-summary.json",
}

required_inputs = [
    value for key, value in PATHS.items()
    if key not in {"sector_candidates", "design_summary"}
]
missing = [str(path) for path in required_inputs if not path.exists()]
assert not missing, f"Missing required inputs: {missing}"

print(f"Repository root: {ROOT}")
print(f"Figures: {FIGURE_DIR}")

In [ ]:
plan = pd.DataFrame(json.loads((ROOT / "implementation-plan.json").read_text()))
pilot_plan_ids = [
    "P009", "P010", "P020", "P021", "P023", "P024", "P025",
    "P026", "P030", "P037", "P053", "P054", "P081", "P082", "P083",
]

display(
    plan.loc[plan["id"].isin(pilot_plan_ids), ["id", "phase", "kind", "item"]]
    .set_index("id")
)

## 2. Inputs and provenance checks

The live availability manifest is metadata only. The AIMS data are an Environment
Agency current-state snapshot, not defence history. The OS line is an undated
provisional chainage and geometry seed, not a dated shoreline observation.

The detailed interactive AIMS investigation is preserved locally at
`notebooks/exploratory/01-label-pilot-context-working.ipynb`. This concise notebook
validates its saved decisions without replaying every inspection step.

In [ ]:
rois = gpd.read_file(PATHS["rois"])
seed_gdf = gpd.read_file(PATHS["seed"])
seed = seed_gdf.geometry.iloc[0]

availability = pd.read_csv(PATHS["availability"], keep_default_na=False)
scene_pool = pd.read_csv(PATHS["scene_pool"], keep_default_na=False)
scene_summary = json.loads(PATHS["scene_summary"].read_text())

aims_inspection = gpd.read_file(PATHS["aims_inspection"])
aims_inspection["asset_id"] = aims_inspection["asset_id"].astype(str)
asset_review = pd.read_csv(
    PATHS["asset_review"], dtype={"asset_id": "string"}, keep_default_na=False
)
functional_review = pd.read_csv(
    PATHS["functional_review"], dtype={"asset_id": "string"}, keep_default_na=False
)
frontage_review = pd.read_csv(PATHS["frontage_review"], keep_default_na=False)
council_review = pd.read_csv(PATHS["council_review"], keep_default_na=False)

assert rois.crs.to_epsg() == 27700
assert seed_gdf.crs.to_epsg() == 27700
assert seed.geom_type == "LineString"
assert functional_review["asset_id"].is_unique
assert frontage_review["frontage_group_id"].is_unique

licence_manifest = json.loads(
    (ROOT / "docs/data-licence-manifest.json").read_text()
)
licence_records = {
    item["id"]: item for item in licence_manifest["datasets"]
}
provenance = pd.DataFrame(
    [
{
    "source": "OS provisional seed",
    "version": licence_records[
        "os_openmap_local_tidal_boundary"
    ]["source"]["source_version"],
    "licence": licence_records[
        "os_openmap_local_tidal_boundary"
    ]["licensing"]["licence"],
    "release_ready": licence_records[
        "os_openmap_local_tidal_boundary"
    ]["publication"]["release_ready"],
},
{
    "source": "EA AIMS",
    "version": licence_records[
        "environment_agency_aims_spatial_flood_defences"
    ]["source"]["source_version"],
    "licence": licence_records[
        "environment_agency_aims_spatial_flood_defences"
    ]["licensing"]["licence"],
    "release_ready": licence_records[
        "environment_agency_aims_spatial_flood_defences"
    ]["publication"]["release_ready"],
},
    ]
)
display(provenance)

## 3. The scene candidate pool has scene-specific fields only

Stage 04 deliberately creates a broad, reproducible metadata pool. It does not yet
choose the final pilot. Defence and morphology are not copied across approximately
5 km ROIs; they will be attached only to local sector–scene combinations.

In [ ]:
forbidden_roi_context = {
    "defence_status", "coastal_regime", "morphology_notes"
}
assert forbidden_roi_context.isdisjoint(scene_pool.columns)
assert "water_level_band" in scene_pool.columns
assert scene_pool["water_level_band"].eq("").all()
assert scene_summary["status"] == "scene_candidate_pool_not_frozen"
assert scene_summary["spatial_context_unit"] == "local_sector_not_roi"

scene_pool_overview = pd.DataFrame(
    {
"candidate_rows": [len(scene_pool)],
"unique_scene_ids": [scene_pool["source_scene_id"].nunique()],
"unique_dates": [
    pd.to_datetime(
            scene_pool["acquisition_time_utc"], format="mixed", utc=True
        ).dt.date.nunique()
],
"roi_count": [scene_pool["roi_id"].nunique()],
"status": [scene_summary["status"]],
    }
)
display(scene_pool_overview)

In [ ]:
eligible_unique = (
    availability.loc[availability["primary_geometry_eligible"]]
    .drop_duplicates(["sensor", "source_scene_id"])
)
sensor_order = ["L5", "L7", "L8", "L9"]
decade_order = ["1990s", "2000s", "2010s", "2020s"]
mission_decade = (
    eligible_unique.groupby(["sensor", "decade"])["source_scene_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=sensor_order, columns=decade_order, fill_value=0)
)

fig, ax = plt.subplots(figsize=(7.2, 3.8))
image = ax.imshow(mission_decade.to_numpy(), cmap="Blues", aspect="auto")
for row in range(mission_decade.shape[0]):
    for column in range(mission_decade.shape[1]):
        value = int(mission_decade.iloc[row, column])
        ax.text(column, row, value, ha="center", va="center", fontsize=9)
ax.set_xticks(range(len(decade_order)), decade_order)
ax.set_yticks(range(len(sensor_order)), sensor_order)
ax.set_xlabel("Acquisition decade")
ax.set_ylabel("Landsat mission")
ax.set_title("Geometry-eligible Landsat scenes available for pilot design")
fig.colorbar(image, ax=ax, label="Unique source scenes", shrink=0.8)
fig.tight_layout()

availability_figure = FIGURE_DIR / "landsat-availability-by-mission-decade.png"
fig.savefig(availability_figure, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {availability_figure}")

## 4. Defence evidence retained for local sector design

The AIMS audit is useful here only to identify known method-test settings. It is not
being completed as a coast-wide covariate at this stage.

The review funnel below makes the filtering traceable. External satellite mapping was
used only as a qualitative cross-check during the working review; no basemap pixels are
retained. Council-named frontages without reusable reviewed geometry remain documented
gaps and are not treated as evidence of an undefended coast.

In [ ]:
aims_files = list((ROOT / "data/raw/aims").glob("*.gpkg"))
assert len(aims_files) == 1, aims_files
aims_path = aims_files[0]
layer_name = pyogrio.list_layers(aims_path)[0][0]

padding_m = 1_000
minx, miny, maxx, maxy = rois.total_bounds
bbox = (
    minx - padding_m,
    miny - padding_m,
    maxx + padding_m,
    maxy + padding_m,
)
aims_bbox = gpd.read_file(aims_path, layer=layer_name, bbox=bbox)
if aims_bbox.crs.to_epsg() != 27700:
    aims_bbox = aims_bbox.to_crs(27700)

local_envelope = rois.geometry.union_all().buffer(padding_m)
aims_local = aims_bbox.loc[
    aims_bbox.geometry.notna()
    & ~aims_bbox.geometry.is_empty
    & aims_bbox.intersects(local_envelope)
].copy()

def clean_text(frame, column):
    return frame[column].fillna("").astype(str).str.strip()

coastal_signal = (
    clean_text(aims_local, "protection_type").isin(["Coastal", "Fluvial/Tidal"])
    | clean_text(aims_local, "water_course_name").isin(["Coast", "Humber"])
    | clean_text(aims_local, "alternative_purpose_1").eq("Erosion Protection")
)
aims_coastal = aims_local.loc[coastal_signal].copy()
natural_types = {"Natural High Ground", "Cliff"}
aims_engineered = aims_coastal.loc[
    ~clean_text(aims_coastal, "asset_sub_type").isin(natural_types)
].copy()

funnel_counts = pd.Series(
    {
"Bounding-box query": len(aims_bbox),
"Buffered study envelope": len(aims_local),
"Coastal signal": len(aims_coastal),
"Possible engineered assets": len(aims_engineered),
"Retained Holderness review": int(
    asset_review["study_area_status"].eq("keep_holderness").sum()
),
"Grouped relevant assets": int(
    functional_review["frontage_group_id"].ne("").sum()
),
    },
    name="records",
)

assert funnel_counts.to_dict() == {
    "Bounding-box query": 266,
    "Buffered study envelope": 94,
    "Coastal signal": 44,
    "Possible engineered assets": 39,
    "Retained Holderness review": 34,
    "Grouped relevant assets": 33,
}
assert functional_review["functional_status"].value_counts().to_dict() == {
    "frontage_defence": 20,
    "integral_frontage_structure": 13,
    "uncertain_function": 1,
}
display(funnel_counts.to_frame())

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.6))
labels = funnel_counts.index.tolist()[::-1]
values = funnel_counts.to_numpy()[::-1]
colors = plt.cm.Blues(np.linspace(0.35, 0.9, len(values)))
bars = ax.barh(labels, values, color=colors, edgecolor="white")
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_xlabel("AIMS records")
ax.set_title("AIMS defence-context review funnel")
ax.spines[["top", "right"]].set_visible(False)
ax.text(
    0.99,
    0.03,
    "Final split: 33 grouped relevant assets + 1 uncertain asset",
    transform=ax.transAxes,
    ha="right",
    fontsize=8,
    color="0.35",
)
fig.tight_layout()

funnel_figure = FIGURE_DIR / "aims-defence-review-flow.png"
fig.savefig(funnel_figure, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {funnel_figure}")

In [ ]:
reviewed_geometry = aims_inspection.merge(
    functional_review[
["asset_id", "functional_status", "frontage_group_id"]
    ],
    on="asset_id",
    how="inner",
)
group_names = frontage_review.set_index("frontage_group_id")[
    "named_location"
].to_dict()
group_colors = {
    "HOL_FR_WITHERNSEA_CANDIDATE": "#d73027",
    "HOL_FR_HORNSEA_CANDIDATE": "#d73027",
    "HOL_FR_BARMSTON_OUTLET_CANDIDATE": "#d73027",
    "HOL_FR_BRIDLINGTON_CANDIDATE": "#d73027",
}

fig, ax = plt.subplots(figsize=(7.2, 11.0))
rois.boundary.plot(ax=ax, color="0.80", linewidth=0.6)
seed_gdf.plot(ax=ax, color="#2166ac", linewidth=1.3)

for group_id, color in group_colors.items():
    group = reviewed_geometry.loc[
reviewed_geometry["frontage_group_id"].eq(group_id)
    ]
    group.plot(ax=ax, color=color, linewidth=2.5)
    point = group.geometry.union_all().centroid
    ax.annotate(
group_names[group_id],
(point.x, point.y),
xytext=(5, 3),
textcoords="offset points",
fontsize=8,
    )

uncertain = reviewed_geometry.loc[
    reviewed_geometry["functional_status"].eq("uncertain_function")
]
uncertain.plot(ax=ax, color="0.35", linewidth=1.8, linestyle="--")
if not uncertain.empty:
    point = uncertain.geometry.union_all().centroid
    ax.annotate(
"Asset 26124: function unresolved",
(point.x, point.y),
xytext=(5, 3),
textcoords="offset points",
fontsize=8,
    )

handles = [
    Line2D([0], [0], color="#2166ac", lw=1.5, label="Provisional OS seed"),
    Line2D([0], [0], color="0.80", lw=1, label="Availability ROIs"),
    Line2D([0], [0], color="#d73027", lw=2.5, label="Reviewed AIMS frontage"),
    Line2D([0], [0], color="0.35", lw=1.8, ls="--", label="Uncertain AIMS asset"),
]
ax.legend(handles=handles, loc="upper right", fontsize=8)
ax.set_title("Reviewed current defence context along Holderness")
ax.set_xlabel("British National Grid easting (m)")
ax.set_ylabel("British National Grid northing (m)")
ax.set_aspect("equal")
fig.text(
    0.01,
    0.01,
    "Working evidence only: AIMS is current-state; group boundaries and OS chainage are provisional.",
    fontsize=7,
    color="0.35",
)
fig.tight_layout(rect=(0, 0.025, 1, 1))

overview_figure = FIGURE_DIR / "reviewed-defence-context-overview.png"
fig.savefig(overview_figure, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {overview_figure}")

## 5. Provisional local sector candidates

The objective is the smallest set that tests materially different extraction
conditions—not geographic coverage for its own sake.

Candidate construction uses the provisional OS chainage only:

- **Withernsea transition:** complete reviewed frontage plus approximately 500 m
  context at each end, rounded outward to the planned 50 m spacing.
- **Coastal-cliff comparison:** a 1 km interval inside long AIMS asset `51041`
  (`Cliff`, `Coastal`) with no reviewed engineered AIMS asset in its 250 m envelope.
  This is not yet evidence of exposed till or definitive undefended status.
- **Barmston outlet:** complete reviewed rock-armour/outlet asset plus approximately
  500 m context at each end.
- **Hornsea south end:** retained as a backup because its dense, overlapping defence
  structures make it a less clean primary transition test.

The three core sectors, approximately 500 m context rule and Hornsea backup were
approved on 2026-08-07. The cliff comparison remains explicitly provisional; ords
remain scene-specific morphology observations.

In [ ]:
def projected_chainage_span(geometry):
    values = [
seed.project(Point(float(x), float(y)))
for x, y in get_coordinates(geometry)
    ]
    return min(values), max(values)

group_rows = []
for group_id, group in reviewed_geometry.loc[
    reviewed_geometry["frontage_group_id"].ne("")
].groupby("frontage_group_id"):
    geometry = group.geometry.union_all()
    start_m, end_m = projected_chainage_span(geometry)
    group_rows.append(
{
    "frontage_group_id": group_id,
    "named_location": group_names[group_id],
    "asset_count": len(group),
    "chainage_start_m": start_m,
    "chainage_end_m": end_m,
    "span_m": end_m - start_m,
}
    )

frontage_spans = (
    pd.DataFrame(group_rows).sort_values("chainage_start_m").reset_index(drop=True)
)
display(frontage_spans.round(1))

In [ ]:
sector_candidates = pd.DataFrame(
    [
{
    "pilot_sector_id": "HOL_PILOT_WITHERNSEA_TRANSITION",
    "role": "core_candidate",
    "context_requirement": "defended_frontage_and_both_ends",
    "chainage_start_m": 13250.0,
    "chainage_end_m": 15450.0,
    "evidence_basis": "Complete reviewed Withernsea AIMS group plus approximately 500 m context at each end.",
    "review_status": "approved_core_sector",
},
{
    "pilot_sector_id": "HOL_PILOT_CLIFF_COMPARISON_CANDIDATE",
    "role": "core_candidate",
    "context_requirement": "coastal_cliff_comparison",
    "chainage_start_m": 36200.0,
    "chainage_end_m": 37200.0,
    "evidence_basis": "Within AIMS coastal cliff asset 51041; no reviewed engineered AIMS asset intersects the 250 m candidate envelope.",
    "review_status": "approved_as_provisional_comparison",
},
{
    "pilot_sector_id": "HOL_PILOT_BARMSTON_OUTLET",
    "role": "core_candidate",
    "context_requirement": "drain_outlet_and_rock_armour",
    "chainage_start_m": 50000.0,
    "chainage_end_m": 51250.0,
    "evidence_basis": "Complete reviewed Barmston outlet AIMS asset plus approximately 500 m context at each end.",
    "review_status": "approved_core_sector",
},
{
    "pilot_sector_id": "HOL_PILOT_HORNSEA_SOUTH_END_BACKUP",
    "role": "backup_candidate",
    "context_requirement": "defence_end_transition_backup",
    "chainage_start_m": 38100.0,
    "chainage_end_m": 39150.0,
    "evidence_basis": "Alternative defence-end test around the southern Hornsea frontage; dense structures reduce interpretability.",
    "review_status": "approved_backup_sector",
},
    ]
)

sector_candidates["sector_length_m"] = (
    sector_candidates["chainage_end_m"]
    - sector_candidates["chainage_start_m"]
)
sector_candidates["provisional_geometry"] = True
sector_candidates["chainage_source"] = "undated_os_geometry_seed"

sector_geometries = []
roi_ids = []
for row in sector_candidates.itertuples(index=False):
    line = substring(seed, row.chainage_start_m, row.chainage_end_m)
    envelope = line.buffer(250, cap_style="flat")
    sector_geometries.append(line)
    roi_ids.append(
"|".join(rois.loc[rois.intersects(envelope), "roi_id"].tolist())
    )

sector_candidates["intersecting_roi_ids"] = roi_ids
sector_gdf = gpd.GeoDataFrame(
    sector_candidates.copy(), geometry=sector_geometries, crs=27700
)

assert (sector_candidates["chainage_start_m"] % 50 == 0).all()
assert (sector_candidates["chainage_end_m"] % 50 == 0).all()
assert sector_candidates["pilot_sector_id"].is_unique
assert not sector_gdf.geometry.is_empty.any()

PATHS["sector_candidates"].parent.mkdir(parents=True, exist_ok=True)
sector_candidates.to_csv(PATHS["sector_candidates"], index=False)
display(sector_candidates)
print(f"Wrote {PATHS['sector_candidates']}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 11.0))
rois.boundary.plot(ax=ax, color="0.85", linewidth=0.6)
seed_gdf.plot(ax=ax, color="0.65", linewidth=1.0)
reviewed_geometry.loc[
    reviewed_geometry["frontage_group_id"].ne("")
].plot(ax=ax, color="#2166ac", linewidth=1.5, alpha=0.75)

sector_colors = {
    "core_candidate": "#d73027",
    "backup_candidate": "#fdae61",
}
for row in sector_gdf.itertuples(index=False):
    gpd.GeoSeries([row.geometry], crs=27700).plot(
ax=ax,
color=sector_colors[row.role],
linewidth=5 if row.role == "core_candidate" else 3,
alpha=0.9,
    )
    midpoint = row.geometry.interpolate(0.5, normalized=True)
    label = row.pilot_sector_id.replace("HOL_PILOT_", "").replace("_", " ")
    ax.annotate(
label,
(midpoint.x, midpoint.y),
xytext=(6, 2),
textcoords="offset points",
fontsize=7.5,
    )

handles = [
    Line2D([0], [0], color="#d73027", lw=5, label="Approved core sector"),
    Line2D([0], [0], color="#fdae61", lw=3, label="Approved backup sector"),
    Line2D([0], [0], color="#2166ac", lw=1.5, label="Reviewed AIMS frontage"),
    Line2D([0], [0], color="0.65", lw=1, label="Provisional OS seed"),
]
ax.legend(handles=handles, loc="upper left", fontsize=8)
ax.set_title("Approved provisional local sectors for the pilot")
ax.set_xlabel("British National Grid easting (m)")
ax.set_ylabel("British National Grid northing (m)")
ax.set_aspect("equal")
fig.text(
    0.01,
    0.01,
    "Approved design: no imagery retrieved; OS chainage and exact defence boundaries remain provisional.",
    fontsize=7,
    color="0.35",
)
fig.tight_layout(rect=(0, 0.025, 1, 1))

sector_figure = FIGURE_DIR / "pilot-sector-candidates.png"
fig.savefig(sector_figure, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {sector_figure}")

## 6. Recorded sector decision

The user approved three core sectors, approximately 500 m of surrounding context
where practicable, and Hornsea south end as a backup on 2026-08-07. Asset `51041`
supports only a provisional coastal-cliff comparison; it is not evidence of exposed
till or a definitively undefended frontage. Ords will be reviewed per scene rather
than assigned as a permanent sector property.

The pilot remains not frozen. Withernsea and Hornsea defence histories are not known
from AIMS; they must not be labelled as defended for every Landsat acquisition date.
Barmston asset `129274` is the only proposed sector asset with an AIMS start date of
`01/01/1990`, which still requires later provenance checking before historical use.

In [ ]:
design_summary = {
    "schema_version": 1,
    "status": "sector_design_approved_scene_design_pending",
    "sector_decision_date": "2026-08-07",
    "purpose": "blind methods validation and rule freezing before full-coast rates",
    "scene_candidate_pool_status": scene_summary["status"],
    "core_sector_count": int(
        sector_candidates["role"].eq("core_candidate").sum()
    ),
    "backup_sector_count": int(
        sector_candidates["role"].eq("backup_candidate").sum()
    ),
    "pending_decisions": [
        "prescribe minimum unique-scene replication",
        "approve sector-scene still-water definition and model-node convention",
        "choose lean scene/date shortlist",
    ],
    "pending_evidence": [
        "FES2022 and GTSM v3 water-level assignment",
        "EA LiDAR independent-validation matching",
        "visual local-cloud and morphology review after retrieval is authorised",
    ],
    "not_authorised": [
        "imagery retrieval",
        "shoreline extraction",
        "full-coast rate calculation",
    ],
    "generated_figures": [
        "outputs/figures/landsat-availability-by-mission-decade.png",
        "outputs/figures/aims-defence-review-flow.png",
        "outputs/figures/reviewed-defence-context-overview.png",
        "outputs/figures/pilot-sector-candidates.png",
    ],
}

PATHS["design_summary"].parent.mkdir(parents=True, exist_ok=True)
PATHS["design_summary"].write_text(
    json.dumps(design_summary, indent=2) + "\n"
)
print(json.dumps(design_summary, indent=2))
print(f"Wrote {PATHS['design_summary']}")

## 7. Next step: scene and water-level evidence

Use the live metadata manifest to propose the smallest date set that jointly covers
Landsat missions, decades and seasons with valid geometry, comparatively low
scene-wide cloud and backups. Water-level balance and independent-validation matching
remain explicit gates before the final pilot is frozen.

No imagery retrieval or shoreline extraction should be added until those remaining
gates have been resolved.

### Figure-use limitation

Figures in `outputs/figures/` are ignored working/report-support material. AIMS is a
current-state snapshot; exact frontage boundaries remain provisional; the OS seed is
undated; ROIs are retrieval units. Required OS/EA attribution and the outstanding
release blockers in `docs/data-licence-manifest.json` must be resolved before external
publication.